# Nearest-Neighbor Case Study

这个notebook直接复用retrieval阶段保存的结果，不重新加载模型：

- `metadata.json`：图片路径、caption、图片和caption的对应关系
- `similarity.npy`：image-text相似度矩阵，shape为`[num_images, num_captions]`

支持两种case：

1. 给定图片，显示输入图片，并找最相近的N条文本
2. 给定文本，显示输入文本，并找最相近的10张图片

注意：这里默认查询对象来自COCO val2017已有图片/caption。如果要输入任意新图片或任意新文本，需要重新加载对应模型抽embedding。

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

plt.rcParams["figure.dpi"] = 130
pd.set_option("display.max_colwidth", 180)


RETRIEVAL_ROOT = Path.cwd() / ".." / "retrieval"

MODEL_DIRS = {
    "clip": "outputs_clip",
    "blip": "outputs_blip",
    "blip2": "outputs_blip2",
}


MODEL = "clip"

In [ ]:
CACHE = {}


def load_case_data(model=MODEL):
    model = model.lower()
    if model not in MODEL_DIRS:
        raise ValueError(f"Unknown model: {model}. Choose from {list(MODEL_DIRS)}")
    if model in CACHE:
        return CACHE[model]

    root = RETRIEVAL_ROOT / MODEL_DIRS[model]
    metadata_path = root / "metadata.json"
    similarity_path = root / "similarity.npy"
    if not metadata_path.exists():
        raise FileNotFoundError(f"Missing {metadata_path}. Run retrieval first.")
    if not similarity_path.exists():
        raise FileNotFoundError(f"Missing {similarity_path}. Run retrieval first.")

    with metadata_path.open("r", encoding="utf-8") as f:
        metadata = json.load(f)
    similarity = np.load(similarity_path)

    data = {
        "model": model,
        "metadata": metadata,
        "similarity": similarity,
        "image_ids": [int(x) for x in metadata["image_ids"]],
        "image_paths": metadata["image_paths"],
        "captions": metadata["captions"],
        "caption_image_indices": [int(x) for x in metadata["caption_image_indices"]],
        "image_to_caption_indices": metadata["image_to_caption_indices"],
    }
    CACHE[model] = data
    return data


def show_dataset_info(model=MODEL):
    data = load_case_data(model)
    display(Markdown(
        f"**Model:** `{data['model']}`  \n"
        f"**Images:** `{len(data['image_paths'])}`  \n"
        f"**Captions:** `{len(data['captions'])}`  \n"
        f"**Similarity shape:** `{data['similarity'].shape}`"
    ))


show_dataset_info(MODEL)

In [ ]:
def resolve_image_index(data, image_index=None, image_id=None, image_path=None):
    if image_index is not None:
        image_index = int(image_index)
        if not 0 <= image_index < len(data["image_paths"]):
            raise IndexError(f"image_index out of range: {image_index}")
        return image_index

    if image_id is not None:
        image_id = int(image_id)
        if image_id not in data["image_ids"]:
            raise ValueError(f"image_id not found: {image_id}")
        return data["image_ids"].index(image_id)

    if image_path is not None:
        image_path = str(Path(image_path).resolve())
        resolved_paths = [str(Path(p).resolve()) for p in data["image_paths"]]
        if image_path not in resolved_paths:
            raise ValueError(f"image_path not found: {image_path}")
        return resolved_paths.index(image_path)

    raise ValueError("Pass one of: image_index, image_id, image_path")


def resolve_caption_index(data, caption_index=None, text=None):
    if caption_index is not None:
        caption_index = int(caption_index)
        if not 0 <= caption_index < len(data["captions"]):
            raise IndexError(f"caption_index out of range: {caption_index}")
        return caption_index

    if text is not None:
        matches = [i for i, cap in enumerate(data["captions"]) if cap == text]
        if not matches:
            partial = [i for i, cap in enumerate(data["captions"]) if text.lower() in cap.lower()]
            preview = [(i, data["captions"][i]) for i in partial[:10]]
            raise ValueError(
                "text不是COCO已有caption的精确匹配。"
                "可以用caption_index查询；或从下面partial matches里挑一个caption_index：\n"
                + "\n".join([f"{i}: {cap}" for i, cap in preview])
            )
        return matches[0]

    raise ValueError("Pass one of: caption_index, text")


def image_gt_captions(data, image_index):
    return [data["captions"][i] for i in data["image_to_caption_indices"][image_index]]


def show_image(path, title=None, width=5):
    image = Image.open(path).convert("RGB")
    h, w = image.height, image.width
    fig_h = max(2.8, width * h / max(w, 1))
    fig, ax = plt.subplots(figsize=(width, fig_h))
    ax.imshow(image)
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10)
    plt.show()


def show_image_grid(image_paths, titles, columns=5, cell_width=3.0):
    n = len(image_paths)
    rows = math.ceil(n / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(columns * cell_width, rows * (cell_width + 0.6)))
    if rows == 1 and columns == 1:
        axes = np.array([[axes]])
    elif rows == 1:
        axes = np.array([axes])
    axes = axes.reshape(rows, columns)

    for ax in axes.ravel():
        ax.axis("off")

    for i, (path, title) in enumerate(zip(image_paths, titles)):
        ax = axes.ravel()[i]
        image = Image.open(path).convert("RGB")
        ax.imshow(image)
        ax.set_title(title, fontsize=8)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 图片 -> 最近文本

用法：传入`image_index`、`image_id`或`image_path`三选一。输出会显示输入图片、该图片的5条GT captions，以及模型检索到的top-N文本。

In [ ]:
def image_to_text_case(model=MODEL, image_index=None, image_id=None, image_path=None, top_n=10):
    data = load_case_data(model)
    image_index = resolve_image_index(data, image_index=image_index, image_id=image_id, image_path=image_path)
    image_id = data["image_ids"][image_index]
    path = data["image_paths"][image_index]

    display(Markdown(f"## Image -> Text Case  \n**Model:** `{model}`  \n**image_index:** `{image_index}`  \n**image_id:** `{image_id}`"))
    show_image(path, title=f"query image: {Path(path).name}", width=5)

    display(Markdown("### Ground Truth Captions"))
    for i, cap in enumerate(image_gt_captions(data, image_index), start=1):
        display(Markdown(f"{i}. {cap}"))

    scores = data["similarity"][image_index]
    top_caption_indices = np.argsort(-scores)[:top_n]
    rows = []
    for rank, cap_idx in enumerate(top_caption_indices, start=1):
        cap_idx = int(cap_idx)
        matched_image_index = data["caption_image_indices"][cap_idx]
        rows.append({
            "rank": rank,
            "score": float(scores[cap_idx]),
            "caption_index": cap_idx,
            "hit_gt_image": matched_image_index == image_index,
            "caption_image_id": data["image_ids"][matched_image_index],
            "caption": data["captions"][cap_idx],
        })

    display(Markdown(f"### Top-{top_n} Retrieved Captions"))
    display(pd.DataFrame(rows))

image_to_text_case(model="clip", image_id=300039, top_n=10)
image_to_text_case(model="blip", image_id=300039, top_n=10)
image_to_text_case(model="blip2", image_id=300039, top_n=10)


## 文本 -> 最近图片

用法：传入`caption_index`，或者传入一个COCO已有caption的精确文本。输出会显示输入文本、它对应的GT图片，以及检索到的top-10图片。

In [ ]:
def text_to_image_case(model=MODEL, caption_index=None, text=None, top_n=10, columns=5):
    data = load_case_data(model)
    caption_index = resolve_caption_index(data, caption_index=caption_index, text=text)
    query_text = data["captions"][caption_index]
    gt_image_index = data["caption_image_indices"][caption_index]
    gt_image_id = data["image_ids"][gt_image_index]

    display(Markdown(f"## Text -> Image Case  \n**Model:** `{model}`  \n**caption_index:** `{caption_index}`  \n**GT image_id:** `{gt_image_id}`"))
    display(Markdown(f"### Query Text\n> {query_text}"))

    display(Markdown("### Ground Truth Image"))
    show_image(data["image_paths"][gt_image_index], title=f"GT image: {Path(data['image_paths'][gt_image_index]).name}", width=5)

    scores = data["similarity"][:, caption_index]
    top_image_indices = np.argsort(-scores)[:top_n]

    image_paths = []
    titles = []
    rows = []
    for rank, image_idx in enumerate(top_image_indices, start=1):
        image_idx = int(image_idx)
        image_paths.append(data["image_paths"][image_idx])
        is_gt = image_idx == gt_image_index
        title = f"#{rank} score={scores[image_idx]:.3f}\nimage_id={data['image_ids'][image_idx]}"
        if is_gt:
            title += "  GT"
        titles.append(title)
        rows.append({
            "rank": rank,
            "score": float(scores[image_idx]),
            "image_index": image_idx,
            "image_id": data["image_ids"][image_idx],
            "hit_gt_image": is_gt,
            "first_gt_caption_of_image": image_gt_captions(data, image_idx)[0],
            "image_path": data["image_paths"][image_idx],
        })

    display(Markdown(f"### Top-{top_n} Retrieved Images"))
    show_image_grid(image_paths, titles, columns=columns)
    display(pd.DataFrame(rows))

text_to_image_case(model=MODEL, text='Two men shake hands at a formal dinner gathering.', top_n=10, columns=5)